# Candidate B: Path Dependence -- Local Full Pilot (RTX 5090)

One notebook, run top to bottom:
1. **Setup + model load** (real `early_stop` package, not a re-transcription)
2. **Timing probe** -- measures real tok/s on this machine
3. **Go/no-go checkpoint** -- prints a projected wall-clock estimate for the full pilot and stops for you to confirm before spending GPU-hours
4. **Full pilot** -- exactly `CANDIDATE_B_PREREGISTRATION.md`'s design: 18 problems (6 per MATH-500 difficulty level), 24 matched pairs (8 per level) x 40 branches/pair, confidence+problem_id stratified sampling, per-stratum + combined Fisher's-method reporting
5. **Final report + save**

Run from the repo root (`research/`) so the `early_stop` package import resolves, or edit `REPO_ROOT` in the next cell.

In [ ]:
import sys, os, time, json

# --- point this at the repo root if running from elsewhere ---
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "scripts" else os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print(f"REPO_ROOT = {REPO_ROOT}")

from early_stop.backend import HFBackend, DEEPSEEK_R1_DISTILL_QWEN_7B_PARAMS_B, probe_gpu
from early_stop.candidate_b_pipeline import (
    CANDIDATE_B_SUFFIX, generate_diverse_trajectories, build_trajectory_prefixes,
    branch_naturally,
)
from early_stop.path_dependence import (
    find_matched_pairs, stratified_sample_pairs, pair_permutation_test, combine_pair_results,
)

MODEL_REPO = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
TEMPERATURE = 0.6

gpu = probe_gpu()
print(gpu.summary())
if not gpu.available:
    raise SystemExit("No CUDA GPU detected -- this notebook needs the local GPU machine.")

In [ ]:
print(f"Loading {MODEL_REPO} ...")
t0 = time.time()
backend = HFBackend(
    model_name=MODEL_REPO,
    temperature=TEMPERATURE,
    param_count_b=DEEPSEEK_R1_DISTILL_QWEN_7B_PARAMS_B,
)
print(f"Model loaded in {time.time() - t0:.1f}s")

import torch
if torch.cuda.is_available():
    print(f"Peak VRAM after load: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GiB")

## Step 2: Timing probe

Real calls through the real backend -- one 2000-token trace, three 48-token forced-extraction calls, one 1500-token branch continuation. This is what the actual pilot's Phase 1/2/3 calls look like, just a handful of them, so the tok/s numbers are real.

In [ ]:
SAMPLE_PROBLEM = (
    "Find the sum of all positive integers $n$ such that $n^2 - 19n + 99$ "
    "is a perfect square. Show your work step by step, then give the final answer."
)

def _tok_per_sec(text, elapsed):
    n = len(backend.tokenizer(text, add_special_tokens=False).input_ids)
    return n / elapsed if elapsed > 0 else float("nan")

timing = {}

print("[1/3] Full trace generation (2000 tokens)...")
t0 = time.time()
trace = backend.generate(SAMPLE_PROBLEM, max_new_tokens=2000)
elapsed = time.time() - t0
timing["trace_2000tok"] = elapsed
print(f"  {elapsed:.1f}s, ~{_tok_per_sec(trace, elapsed):.1f} tok/s")

print("\n[2/3] Forced-extraction calls (48 tokens x 3)...")
prefix_text = SAMPLE_PROBLEM + "\n\n" + trace[: len(trace) // 2]
extraction_times = []
for i in range(3):
    t0 = time.time()
    scored = backend.forced_extract_with_scores(prefix_text, CANDIDATE_B_SUFFIX, max_new_tokens=48)
    elapsed = time.time() - t0
    extraction_times.append(elapsed)
    print(f"  call {i+1}: {elapsed:.2f}s (confidence={scored.top_token_confidence:.3f}, entropy={scored.entropy_bits:.2f} bits)")
timing["forced_extract_48tok"] = sum(extraction_times) / len(extraction_times)

print("\n[3/3] Branch continuation (1500 tokens)...")
t0 = time.time()
branch = backend.continue_generate(prefix_text, max_new_tokens=1500)
elapsed = time.time() - t0
timing["branch_1500tok"] = elapsed
print(f"  {elapsed:.1f}s, ~{_tok_per_sec(branch, elapsed):.1f} tok/s")

backend.free_vram()

## Step 3: Go/no-go checkpoint

Projects wall-clock for the full pre-registered pilot (18 problems x 5 trajectories, 24 pairs x 40 branches/pair) from the timing probe's real numbers. **Read the projection, then only run the cells below it if you actually want to commit the GPU time now.**

In [ ]:
N_PROBLEMS_TOTAL = 18
K_TRAJECTORIES_PER_PROBLEM = 5
N_PAIRS_PER_STRATUM = 8
N_STRATA = 3  # MATH-500 levels 1, 2, 3
N_BRANCHES_PER_PAIR = 40

n_trajectories = N_PROBLEMS_TOTAL * K_TRAJECTORIES_PER_PROBLEM
avg_prefixes_per_traj = 32  # observed average across the micro-pilot (1271 prefixes / 40 trajectories)
n_scoring_calls = n_trajectories * avg_prefixes_per_traj
n_pairs_total = N_PAIRS_PER_STRATUM * N_STRATA
n_branch_calls = n_pairs_total * N_BRANCHES_PER_PAIR * 2  # both sides of each pair

phase1_hours = (n_trajectories * timing["trace_2000tok"]) / 3600
phase2_hours = (n_scoring_calls * timing["forced_extract_48tok"]) / 3600
phase3_hours = (n_branch_calls * timing["branch_1500tok"]) / 3600
total_hours = phase1_hours + phase2_hours + phase3_hours

print("PROJECTION onto CANDIDATE_B_PREREGISTRATION.md's full pilot design:")
print(f"  Phase 1 ({n_trajectories} trajectories x {timing['trace_2000tok']:.1f}s):        ~{phase1_hours:.1f}h")
print(f"  Phase 2 (~{n_scoring_calls} scoring calls x {timing['forced_extract_48tok']:.2f}s): ~{phase2_hours:.1f}h")
print(f"  Phase 3 ({n_branch_calls} branches x {timing['branch_1500tok']:.1f}s):          ~{phase3_hours:.1f}h")
print(f"  TOTAL projected wall-clock:                       ~{total_hours:.1f}h")
print("\n(Projected from 5 real calls -- real runs vary with problem/trajectory length.")
print(" Treat as a planning estimate, not a guarantee.)")

# RUN_FULL_PILOT = False  # <-- flip to True only after reading the projection above
# if not RUN_FULL_PILOT:
#     raise SystemExit("Stopped at the checkpoint. Set RUN_FULL_PILOT = True above and re-run this cell to proceed.")

## Step 4a: Phase 1 -- diverse trajectories + phi(p_t) scoring, 18 problems x 5 trajectories

Loads MATH-500, keeps 6 problems per level (1, 2, 3), generates 5 diverse trajectories per problem, scores every step boundary. Per-item printing so a crash partway through loses nothing already computed.

In [ ]:
from datasets import load_dataset

PROBLEMS_PER_LEVEL = 6
OUTPUT_DIR = os.path.join(REPO_ROOT, "candidate_b_full_pilot_local")
os.makedirs(OUTPUT_DIR, exist_ok=True)

math_ds = load_dataset("HuggingFaceH4/MATH-500", split="test")

problems_by_level = {}
for lvl in ("1", "2", "3"):
    subset = math_ds.filter(lambda r, lvl=lvl: str(r["level"]) == lvl)
    problems_by_level[lvl] = subset.select(range(min(PROBLEMS_PER_LEVEL, len(subset))))
    print(f"level {lvl}: {len(problems_by_level[lvl])} problems selected")

assert all(len(v) == PROBLEMS_PER_LEVEL for v in problems_by_level.values()), (
    "a MATH-500 level has fewer than PROBLEMS_PER_LEVEL rows available -- reduce PROBLEMS_PER_LEVEL"
)

In [ ]:
trajectory_prefixes_by_level = {"1": [], "2": [], "3": []}
trace_steps_cache = {}  # (problem_id, trajectory_id) -> list[str] step texts, for Phase 3 real-prefix reconstruction

from early_stop.segmentation import split_into_steps, step_prefix

global_pi = 0
for lvl, rows in problems_by_level.items():
    print(f"\n{'='*70}\nLEVEL {lvl}\n{'='*70}")
    for row in rows:
        problem_id = f"prob{global_pi}"
        global_pi += 1
        base_prompt = row["problem"]
        print(f"\n--- {problem_id} (level {lvl}) ---")
        traces = generate_diverse_trajectories(backend, base_prompt, k=K_TRAJECTORIES_PER_PROBLEM, max_new_tokens=2000)
        for ti, trace_text in enumerate(traces):
            traj_id = f"traj_{ti}"
            t0 = time.time()
            tp = build_trajectory_prefixes(backend, problem_id, traj_id, base_prompt, trace_text, max_new_tokens=48)
            trajectory_prefixes_by_level[lvl].append(tp)
            trace_steps_cache[(problem_id, traj_id)] = split_into_steps(trace_text)
            print(f"  {traj_id}: {len(tp.prefixes)} scored, {tp.n_unparseable} unparseable, {time.time()-t0:.1f}s")

        # checkpoint after every problem in case of a long-run crash
        with open(os.path.join(OUTPUT_DIR, "phase1_checkpoint.json"), "w") as f:
            json.dump({
                lv: [
                    {"problem_id": tp.problem_id, "trajectory_id": tp.trajectory_id,
                     "n_prefixes": len(tp.prefixes), "n_unparseable": tp.n_unparseable}
                    for tp in tps
                ] for lv, tps in trajectory_prefixes_by_level.items()
            }, f, indent=2)

total_prefixes = sum(len(tp.prefixes) for tps in trajectory_prefixes_by_level.values() for tp in tps)
total_unparseable = sum(tp.n_unparseable for tps in trajectory_prefixes_by_level.values() for tp in tps)
print(f"\nPhase 1 done. Total scored prefixes: {total_prefixes} | unparseable: {total_unparseable}")

## Step 4b: Phase 2 -- L1/L2/L3 matched pairs, PER STRATUM (level), then confidence+problem_id-stratified sampling down to 8 pairs/level

Per `CANDIDATE_B_PREREGISTRATION.md` §3: matching and sampling are run independently per difficulty level, not pooled, so one level can't crowd out another's budget.

In [ ]:
CONFIDENCE_TOL = 0.10
ENTROPY_TOL = 0.15
POSITION_TOL = 0.15

sampled_pairs_by_level = {}  # level -> {matchlevel: [pairs]}
for lvl, tps in trajectory_prefixes_by_level.items():
    all_prefixes = [p for tp in tps for p in tp.prefixes]
    sampled_pairs_by_level[lvl] = {}
    print(f"\nlevel {lvl} ({len(all_prefixes)} prefixes):")
    for match_level in ("L1", "L2", "L3"):
        pairs = find_matched_pairs(
            all_prefixes, match_level,
            confidence_tol=CONFIDENCE_TOL, entropy_tol=ENTROPY_TOL, position_tol=POSITION_TOL,
        )
        level_seed = {"L1": 1, "L2": 2, "L3": 3}[match_level]  # NOT hash() -- non-deterministic per-process
        sampled = stratified_sample_pairs(pairs, budget=N_PAIRS_PER_STRATUM, seed=level_seed)
        sampled_pairs_by_level[lvl][match_level] = sampled
        print(f"  {match_level}: {len(pairs)} matched, sampled {len(sampled)} for branching")

## Step 4c: Phase 3 -- natural branching (N=40/side, NO forced suffix) + per-pair permutation test

Only L1 pairs are branched (matches the pre-registration's headline design -- L2/L3 matched-pair COUNTS are still reported above for comparison, but branching budget is spent on L1 per §4's "all three levels tested" being about reporting which level a finding appears at, not branching separately at each -- see note below if you want L2/L3 branched too).

In [ ]:
# NOTE: CANDIDATE_B_PREREGISTRATION.md §4 says all three matching levels are
# reported. Branching 24 pairs x 3 levels x 40 branches would triple Phase 3's
# GPU cost beyond what §6's cost accounting assumed. Default here branches
# ONLY L1 (the most permissive / highest-pair-count level, closest to what a
# pure answer-stability early-stopping method sees) to match §6's budget.
# Set BRANCH_LEVELS = ("L1", "L2", "L3") to branch all three instead (~3x cost).
BRANCH_LEVELS = ("L1",)
print(f"Branching levels: {BRANCH_LEVELS}")

In [ ]:
def _reconstruct_prefix_text(prefix):
    steps = trace_steps_cache[(prefix.problem_id, prefix.trajectory_id)]
    return step_prefix(steps, prefix.step_index)

pair_results_by_level = {}  # level -> {matchlevel: [PairPermutationResult]}
raw_rows = []  # for the final CSV/report

# map problem_id -> base_prompt text, needed by branch_naturally
problem_id_to_prompt = {}
_gi = 0
for lvl, rows in problems_by_level.items():
    for row in rows:
        problem_id_to_prompt[f"prob{_gi}"] = row["problem"]
        _gi += 1

for lvl in ("1", "2", "3"):
    pair_results_by_level[lvl] = {}
    for match_level in BRANCH_LEVELS:
        pairs = sampled_pairs_by_level[lvl][match_level]
        print(f"\n{'='*70}\nlevel {lvl}, {match_level}: branching {len(pairs)} pairs, N={N_BRANCHES_PER_PAIR}/side\n{'='*70}")
        results = []
        for a, b in pairs:
            base_prompt = problem_id_to_prompt[a.problem_id]
            prefix_text_a = _reconstruct_prefix_text(a)
            prefix_text_b = _reconstruct_prefix_text(b)
            t0 = time.time()
            branch_set_a = branch_naturally(backend, base_prompt, prefix_text_a, n=N_BRANCHES_PER_PAIR, max_new_tokens=1500)
            branch_set_b = branch_naturally(backend, base_prompt, prefix_text_b, n=N_BRANCHES_PER_PAIR, max_new_tokens=1500)
            try:
                result = pair_permutation_test(branch_set_a, branch_set_b, min_parse_rate=0.80, reps=2000, seed=0)
                results.append(result)
                print(f"  {a.problem_id}: PDI={result.observed_pdi:.3f} null={result.null_mean:.3f} "
                      f"p={result.p_value:.4f} (n_a={result.n_a} n_b={result.n_b})  [{time.time()-t0:.1f}s]")
                raw_rows.append({
                    "level": match_level, "difficulty": lvl, "problem_id": a.problem_id,
                    "observed_pdi": result.observed_pdi, "null_mean": result.null_mean,
                    "p_value": result.p_value, "effect_size": result.effect_size,
                    "n_a": result.n_a, "n_b": result.n_b,
                })
            except ValueError as e:
                print(f"  {a.problem_id}: SKIPPED ({e})")
        pair_results_by_level[lvl][match_level] = results

        # checkpoint after every (difficulty, match_level) group
        with open(os.path.join(OUTPUT_DIR, "phase3_checkpoint.json"), "w") as f:
            json.dump(raw_rows, f, indent=2)

## Step 5: Final report -- per-stratum and combined, per CANDIDATE_B_PREREGISTRATION.md §5's decision rule

In [ ]:
import pandas as pd

df = pd.DataFrame(raw_rows)
print("##############################################################################")
print("FULL PILOT REPORT")
print("##############################################################################\n")

print("1. PHASE 1 SUMMARY:")
print(f"   Total scored prefixes: {total_prefixes} | unparseable: {total_unparseable} "
      f"({100*total_unparseable/max(1,total_prefixes+total_unparseable):.1f}%)\n")

print("2. PER-STRATUM RESULTS (per CANDIDATE_B_PREREGISTRATION.md §5):")
all_combined = []
for lvl in ("1", "2", "3"):
    for match_level in BRANCH_LEVELS:
        results = pair_results_by_level[lvl].get(match_level, [])
        if not results:
            print(f"   difficulty={lvl} {match_level}: no results (0 valid pairs)")
            continue
        combined = combine_pair_results(results, alpha=0.05)
        all_combined.append((lvl, match_level, combined))
        print(f"   difficulty={lvl} {match_level}: n_pairs={combined.n_pairs} "
              f"Fisher's p={combined.combined_p_value:.4f} "
              f"mean_effect={combined.mean_effect_size:.3f} "
              f"frac_sig={combined.frac_individually_significant:.2f}")

print("\n3. OVERALL (Fisher's method combining all strata' pair-level p-values):")
all_results_flat = [r for lvl in pair_results_by_level.values() for rs in lvl.values() for r in rs]
if all_results_flat:
    overall = combine_pair_results(all_results_flat, alpha=0.05)
    verdict = "REJECT H0 (path dependence detected)" if overall.combined_p_value < 0.05 else "FAIL TO REJECT (no detected path dependence at this scale)"
    print(f"   n_pairs={overall.n_pairs} combined_p={overall.combined_p_value:.4f} "
          f"mean_effect={overall.mean_effect_size:.3f}")
    print(f"   VERDICT: {verdict}")
else:
    print("   No valid pair results to combine.")

df.to_csv(os.path.join(OUTPUT_DIR, "pair_results.csv"), index=False)
print(f"\nSaved: {os.path.join(OUTPUT_DIR, 'pair_results.csv')}")
df